# Chapter 10.5: Feature Matrix - What Each System Does Best

This notebook provides a comprehensive feature comparison between **vLLM** and **SGLang**, covering quantization, multi-modal support, speculative decoding, constrained decoding, LoRA serving, and SGLang's unique frontend DSL.

## Learning Objectives
- Build a complete feature comparison matrix
- Understand unique strengths of each system
- Evaluate systems for specific use cases
- Test and compare unique features

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part10/chapter_10.5_features.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part10/chapter_10.5_features.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
# Setup
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np
from dataclasses import dataclass
from typing import List, Dict, Optional
import json

plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
print("Environment ready for feature comparison.")

## 1. Complete Feature Comparison Matrix

In [ ]:
# Demo 1: Comprehensive feature matrix

feature_matrix = {
    'Core Serving': {
        'Continuous Batching': {'vLLM': 'Yes', 'SGLang': 'Yes', 'notes': 'Both support iteration-level batching'},
        'Chunked Prefill': {'vLLM': 'Yes', 'SGLang': 'Yes', 'notes': 'Reduces head-of-line blocking'},
        'Prefix Caching': {'vLLM': 'Yes (APC, block-level)', 'SGLang': 'Yes (RadixAttention, token-level)', 'notes': 'SGLang more fine-grained'},
        'Streaming Output': {'vLLM': 'Yes', 'SGLang': 'Yes', 'notes': 'SSE-based streaming'},
        'OpenAI-compatible API': {'vLLM': 'Yes (comprehensive)', 'SGLang': 'Yes (comprehensive)', 'notes': 'Both have excellent API coverage'},
    },
    'Quantization': {
        'GPTQ': {'vLLM': 'Yes', 'SGLang': 'Yes', 'notes': ''},
        'AWQ': {'vLLM': 'Yes', 'SGLang': 'Yes', 'notes': ''},
        'GGUF': {'vLLM': 'Yes', 'SGLang': 'Limited', 'notes': 'vLLM has broader GGUF support'},
        'SqueezeLLM': {'vLLM': 'Yes', 'SGLang': 'No', 'notes': 'vLLM exclusive'},
        'FP8': {'vLLM': 'Yes', 'SGLang': 'Yes', 'notes': 'Both support FP8 on Hopper GPUs'},
        'INT8 W8A8': {'vLLM': 'Yes', 'SGLang': 'Yes', 'notes': ''},
        'INT4 W4A16': {'vLLM': 'Yes', 'SGLang': 'Yes', 'notes': 'Via GPTQ/AWQ'},
        'Marlin Kernels': {'vLLM': 'Yes', 'SGLang': 'Yes', 'notes': 'Fast W4A16 kernels'},
    },
    'Multi-Modal': {
        'Vision-Language Models': {'vLLM': 'Yes (LLaVA, etc.)', 'SGLang': 'Yes (LLaVA, etc.)', 'notes': ''},
        'Image Input': {'vLLM': 'Yes', 'SGLang': 'Yes', 'notes': ''},
        'Video Input': {'vLLM': 'Limited', 'SGLang': 'Yes', 'notes': 'SGLang supports more video models'},
        'Audio Input': {'vLLM': 'Limited', 'SGLang': 'Limited', 'notes': 'Both emerging'},
        'Multi-image': {'vLLM': 'Yes', 'SGLang': 'Yes', 'notes': ''},
    },
    'Advanced Decoding': {
        'Beam Search': {'vLLM': 'Yes (CoW)', 'SGLang': 'Limited', 'notes': 'vLLM has better beam search support'},
        'Speculative Decoding': {'vLLM': 'Yes (multiple methods)', 'SGLang': 'Yes (Eagle)', 'notes': 'vLLM supports more draft model types'},
        'Constrained Decoding': {'vLLM': 'Yes (Outlines)', 'SGLang': 'Yes (jump-forward)', 'notes': 'SGLang faster with jump-ahead'},
        'JSON Schema Decoding': {'vLLM': 'Yes', 'SGLang': 'Yes (faster)', 'notes': 'SGLang jump-forward gives big speedup'},
        'Regex Decoding': {'vLLM': 'Yes (via Outlines)', 'SGLang': 'Yes (native)', 'notes': ''},
        'Logprobs': {'vLLM': 'Yes', 'SGLang': 'Yes', 'notes': ''},
    },
    'Distributed Serving': {
        'Tensor Parallelism': {'vLLM': 'Yes', 'SGLang': 'Yes', 'notes': ''},
        'Pipeline Parallelism': {'vLLM': 'Yes', 'SGLang': 'Yes', 'notes': ''},
        'Data Parallelism': {'vLLM': 'Manual', 'SGLang': 'Yes (built-in DP)', 'notes': 'SGLang has DP controller'},
        'Multi-Node': {'vLLM': 'Yes (Ray)', 'SGLang': 'Yes (custom)', 'notes': 'vLLM uses Ray; SGLang uses custom'},
        'Disaggregated Prefill/Decode': {'vLLM': 'Experimental', 'SGLang': 'Yes', 'notes': 'SGLang more mature'},
    },
    'Adapter Serving': {
        'LoRA': {'vLLM': 'Yes (multi-LoRA)', 'SGLang': 'Yes', 'notes': 'Both support serving multiple LoRAs'},
        'QLoRA': {'vLLM': 'Yes', 'SGLang': 'Yes', 'notes': ''},
        'Dynamic LoRA Loading': {'vLLM': 'Yes', 'SGLang': 'Yes', 'notes': ''},
        'LoRA Batching': {'vLLM': 'Yes (SGMV)', 'SGLang': 'Yes', 'notes': 'Batch requests across LoRAs'},
    },
    'Frontend & Programming': {
        'Frontend DSL': {'vLLM': 'No', 'SGLang': 'Yes (SGLang language)', 'notes': 'Major SGLang differentiator'},
        'Fork/Join': {'vLLM': 'No', 'SGLang': 'Yes', 'notes': 'Parallel generation in DSL'},
        'Choices/Select': {'vLLM': 'No', 'SGLang': 'Yes', 'notes': 'Constrained selection in DSL'},
        'Batch Inference API': {'vLLM': 'Yes (LLM class)', 'SGLang': 'Yes (Engine class)', 'notes': ''},
        'Chat Template': {'vLLM': 'Yes', 'SGLang': 'Yes', 'notes': ''},
    },
}

# Print the matrix
for category, features in feature_matrix.items():
    print(f"\n{'='*90}")
    print(f" {category}")
    print(f"{'='*90}")
    print(f"{'Feature':<30} {'vLLM':<25} {'SGLang':<25}")
    print(f"{'-'*80}")
    for feature, details in features.items():
        print(f"{feature:<30} {details['vLLM']:<25} {details['SGLang']:<25}")
        if details['notes']:
            print(f"{'':30} Note: {details['notes']}")

In [ ]:
# Demo 2: Visual feature heatmap

def plot_feature_heatmap():
    """Create a visual heatmap of feature support."""
    # Convert to numerical scores: 0=No, 1=Limited, 2=Yes, 3=Best
    def score(text):
        text = text.lower()
        if 'no' in text:
            return 0
        elif 'limited' in text:
            return 1
        elif 'yes' in text and ('faster' in text or 'better' in text or 'native' in text):
            return 3
        elif 'yes' in text:
            return 2
        return 1
    
    categories = []
    features_list = []
    vllm_scores = []
    sglang_scores = []
    
    for cat, features in feature_matrix.items():
        for feat, details in features.items():
            categories.append(cat)
            features_list.append(feat)
            vllm_scores.append(score(details['vLLM']))
            sglang_scores.append(score(details['SGLang']))
    
    fig, ax = plt.subplots(figsize=(10, 16))
    
    data = np.array([vllm_scores, sglang_scores]).T
    im = ax.imshow(data, cmap='RdYlGn', aspect='auto', vmin=0, vmax=3)
    
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['vLLM', 'SGLang'], fontsize=14, fontweight='bold')
    ax.set_yticks(range(len(features_list)))
    ax.set_yticklabels(features_list, fontsize=9)
    
    # Add text annotations
    for i in range(len(features_list)):
        for j, val in enumerate([vllm_scores[i], sglang_scores[i]]):
            labels = {0: 'No', 1: 'Limited', 2: 'Yes', 3: 'Best'}
            ax.text(j, i, labels[val], ha='center', va='center',
                   fontsize=8, fontweight='bold',
                   color='white' if val == 0 else 'black')
    
    # Add category separators
    prev_cat = ''
    for i, cat in enumerate(categories):
        if cat != prev_cat:
            if prev_cat:
                ax.axhline(y=i-0.5, color='black', linewidth=2)
            ax.text(-0.7, i, cat, fontsize=8, fontweight='bold',
                   rotation=0, ha='right', va='center', color='#333')
            prev_cat = cat
    
    ax.set_title('Feature Support Heatmap: vLLM vs SGLang', fontsize=14, fontweight='bold')
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2, 3], shrink=0.5)
    cbar.set_ticklabels(['No', 'Limited', 'Yes', 'Best'])
    
    plt.tight_layout()
    plt.savefig('feature_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_feature_heatmap()

In [ ]:
# Demo 3: Feature category comparison radar

def plot_category_radar():
    """Radar chart comparing feature categories."""
    categories = list(feature_matrix.keys())
    
    # Count supported features per category
    def count_support(cat, system):
        features = feature_matrix[cat]
        total = len(features)
        supported = sum(1 for f in features.values()
                       if 'yes' in f[system].lower())
        best = sum(1 for f in features.values()
                  if 'faster' in f[system].lower() or 'better' in f[system].lower()
                  or 'native' in f[system].lower())
        return (supported + best * 0.5) / total * 10
    
    vllm_scores = [count_support(c, 'vLLM') for c in categories]
    sglang_scores = [count_support(c, 'SGLang') for c in categories]
    
    # Short category names
    short_names = ['Core\nServing', 'Quantization', 'Multi\nModal',
                   'Advanced\nDecoding', 'Distributed', 'Adapters',
                   'Frontend\n& DSL']
    
    num_vars = len(categories)
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]
    
    vllm_plot = vllm_scores + vllm_scores[:1]
    sglang_plot = sglang_scores + sglang_scores[:1]
    
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
    ax.fill(angles, vllm_plot, alpha=0.15, color='#2196F3')
    ax.plot(angles, vllm_plot, 'o-', linewidth=2, color='#2196F3', label='vLLM')
    ax.fill(angles, sglang_plot, alpha=0.15, color='#4CAF50')
    ax.plot(angles, sglang_plot, 's-', linewidth=2, color='#4CAF50', label='SGLang')
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(short_names, fontsize=10)
    ax.set_ylim(0, 10)
    ax.set_title('Feature Coverage by Category', fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.2, 1.1), fontsize=12)
    
    plt.tight_layout()
    plt.show()

plot_category_radar()

## 2. Deep Dive: Quantization Support

In [ ]:
# Demo 4: Quantization comparison

quant_methods = {
    'GPTQ 4-bit': {'vLLM_support': True, 'SGLang_support': True,
                    'vLLM_perf': 85, 'SGLang_perf': 85},
    'AWQ 4-bit': {'vLLM_support': True, 'SGLang_support': True,
                   'vLLM_perf': 90, 'SGLang_perf': 88},
    'Marlin 4-bit': {'vLLM_support': True, 'SGLang_support': True,
                      'vLLM_perf': 95, 'SGLang_perf': 93},
    'FP8 E4M3': {'vLLM_support': True, 'SGLang_support': True,
                  'vLLM_perf': 92, 'SGLang_perf': 93},
    'INT8 W8A8': {'vLLM_support': True, 'SGLang_support': True,
                   'vLLM_perf': 88, 'SGLang_perf': 87},
    'SqueezeLLM': {'vLLM_support': True, 'SGLang_support': False,
                    'vLLM_perf': 82, 'SGLang_perf': 0},
    'GGUF Q4_0': {'vLLM_support': True, 'SGLang_support': False,
                   'vLLM_perf': 75, 'SGLang_perf': 0},
    'GGUF Q5_K': {'vLLM_support': True, 'SGLang_support': False,
                   'vLLM_perf': 78, 'SGLang_perf': 0},
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Support matrix
methods = list(quant_methods.keys())
vllm_support = [q['vLLM_support'] for q in quant_methods.values()]
sglang_support = [q['SGLang_support'] for q in quant_methods.values()]

x = np.arange(len(methods))
width = 0.35
ax1.barh(x - width/2, [1 if s else 0 for s in vllm_support], width,
        label='vLLM', color='#2196F3', alpha=0.8)
ax1.barh(x + width/2, [1 if s else 0 for s in sglang_support], width,
        label='SGLang', color='#4CAF50', alpha=0.8)
ax1.set_yticks(x)
ax1.set_yticklabels(methods)
ax1.set_xlabel('Supported (1=Yes, 0=No)')
ax1.set_title('Quantization Support Matrix', fontweight='bold')
ax1.legend()

# Performance comparison (where both support)
both_supported = [m for m, q in quant_methods.items()
                  if q['vLLM_support'] and q['SGLang_support']]
vllm_perf = [quant_methods[m]['vLLM_perf'] for m in both_supported]
sglang_perf = [quant_methods[m]['SGLang_perf'] for m in both_supported]

x2 = np.arange(len(both_supported))
ax2.bar(x2 - width/2, vllm_perf, width, label='vLLM', color='#2196F3')
ax2.bar(x2 + width/2, sglang_perf, width, label='SGLang', color='#4CAF50')
ax2.set_xticks(x2)
ax2.set_xticklabels(both_supported, rotation=30, ha='right')
ax2.set_ylabel('Relative Performance (%)')
ax2.set_title('Quantization Performance (where both support)', fontweight='bold')
ax2.legend()
ax2.set_ylim(70, 100)

plt.tight_layout()
plt.show()

print(f"vLLM supports {sum(vllm_support)}/{len(methods)} quantization methods")
print(f"SGLang supports {sum(sglang_support)}/{len(methods)} quantization methods")

## 3. Deep Dive: SGLang Frontend DSL (Unique Advantage)

In [ ]:
# Demo 5: SGLang DSL capabilities showcase

print("SGLang Frontend DSL - Capabilities Not Available in vLLM")
print("=" * 60)

# Example 1: Multi-turn with automatic prefix sharing
dsl_example_1 = '''
# SGLang: Multi-turn chat with automatic KV cache reuse
import sglang as sgl

@sgl.function
def multi_turn_chat(s, questions):
    s += sgl.system("You are a helpful assistant.")
    for q in questions:
        s += sgl.user(q)
        s += sgl.assistant(sgl.gen("response", max_tokens=256))
    # Automatic KV cache sharing between turns!
'''

# Example 2: Fork/Join for parallel generation
dsl_example_2 = '''
# SGLang: Fork/Join - generate multiple completions in parallel
@sgl.function
def parallel_reasoning(s, question):
    s += f"Question: {question}\\n"
    # Fork into 3 parallel reasoning chains
    forks = s.fork(3)
    for i, f in enumerate(forks):
        f += f"Approach {i+1}: "
        f += sgl.gen("reasoning", max_tokens=200)
    # Join and synthesize
    s += sgl.gen("final_answer", max_tokens=100)
'''

# Example 3: Constrained decoding with choices
dsl_example_3 = '''
# SGLang: Native constrained decoding with choices
@sgl.function
def structured_extraction(s, text):
    s += f"Extract sentiment from: {text}\\n"
    s += "Sentiment: " + sgl.gen("sentiment",
                                   choices=["positive", "negative", "neutral"])
    s += "\\nConfidence: " + sgl.gen("confidence",
                                       choices=["high", "medium", "low"])
    s += "\\nReason: " + sgl.gen("reason", max_tokens=50)
'''

# Example 4: JSON schema generation
dsl_example_4 = '''
# SGLang: JSON schema with jump-forward optimization
@sgl.function
def extract_entity(s, text):
    s += f"Extract entities from: {text}\\n"
    s += sgl.gen("result", max_tokens=200,
                  regex=r'\\{"name": "[^"]+", "type": "(person|org|loc)", "confidence": 0\\.\\d+\\}')
'''

examples = [
    ("Multi-turn with Prefix Reuse", dsl_example_1),
    ("Fork/Join Parallel Generation", dsl_example_2),
    ("Constrained Choices", dsl_example_3),
    ("JSON Schema with Jump-Forward", dsl_example_4),
]

for name, code in examples:
    print(f"\n--- {name} ---")
    print(code)

print("\n" + "=" * 60)
print("vLLM Equivalent: These patterns require manual orchestration")
print("with multiple API calls, losing prefix cache benefits.")

In [ ]:
# Demo 6: Constrained decoding speed comparison

def compare_constrained_decoding():
    """Compare constrained decoding performance."""
    
    # Simulated benchmarks for constrained JSON generation
    schema_complexities = ['Simple\n(3 fields)', 'Medium\n(8 fields)',
                            'Complex\n(20 fields)', 'Nested\n(3 levels)']
    
    # Tokens per second for JSON generation
    vllm_outlines = [45, 32, 18, 12]  # vLLM with Outlines
    sglang_jumpforward = [48, 42, 35, 28]  # SGLang with jump-forward
    unconstrained = [55, 55, 55, 55]  # Baseline without constraints
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    x = np.arange(len(schema_complexities))
    width = 0.25
    
    ax1.bar(x - width, unconstrained, width, label='Unconstrained', color='#9E9E9E')
    ax1.bar(x, vllm_outlines, width, label='vLLM + Outlines', color='#2196F3')
    ax1.bar(x + width, sglang_jumpforward, width, label='SGLang jump-forward', color='#4CAF50')
    ax1.set_xticks(x)
    ax1.set_xticklabels(schema_complexities)
    ax1.set_ylabel('Tokens/second')
    ax1.set_title('Constrained JSON Generation Speed', fontweight='bold')
    ax1.legend()
    
    # Overhead comparison
    vllm_overhead = [(u-v)/u*100 for u, v in zip(unconstrained, vllm_outlines)]
    sglang_overhead = [(u-s)/u*100 for u, s in zip(unconstrained, sglang_jumpforward)]
    
    ax2.plot(schema_complexities, vllm_overhead, 'o-', color='#2196F3',
            label='vLLM overhead', linewidth=2, markersize=8)
    ax2.plot(schema_complexities, sglang_overhead, 's-', color='#4CAF50',
            label='SGLang overhead', linewidth=2, markersize=8)
    ax2.set_ylabel('Overhead vs Unconstrained (%)')
    ax2.set_title('Constrained Decoding Overhead', fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nSGLang's jump-forward optimization:")
    print("  - Skips tokens that are deterministic from the schema")
    print("  - Can 'jump' over JSON delimiters, field names, etc.")
    print("  - Advantage grows with schema complexity")

compare_constrained_decoding()

In [ ]:
# Demo 7: Speculative decoding comparison

def compare_speculative_decoding():
    """Compare speculative decoding capabilities."""
    
    spec_methods = {
        'Draft Model (small LLM)': {
            'vLLM': 'Yes', 'SGLang': 'Yes',
            'speedup_range': '1.3x - 2.0x'
        },
        'Eagle (feature-level draft)': {
            'vLLM': 'Experimental', 'SGLang': 'Yes',
            'speedup_range': '1.5x - 2.5x'
        },
        'Medusa (multi-head)': {
            'vLLM': 'Yes', 'SGLang': 'Limited',
            'speedup_range': '1.4x - 2.2x'
        },
        'Prompt Lookup': {
            'vLLM': 'Yes', 'SGLang': 'No',
            'speedup_range': '1.2x - 1.8x'
        },
        'N-gram Draft': {
            'vLLM': 'Yes', 'SGLang': 'No',
            'speedup_range': '1.1x - 1.5x'
        },
    }
    
    print("Speculative Decoding Methods Comparison")
    print(f"{'Method':<30} {'vLLM':<15} {'SGLang':<15} {'Speedup Range':<15}")
    print('=' * 75)
    for method, details in spec_methods.items():
        print(f"{method:<30} {details['vLLM']:<15} {details['SGLang']:<15} {details['speedup_range']:<15}")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(12, 5))
    methods = list(spec_methods.keys())
    
    # Parse speedup ranges
    for i, (method, details) in enumerate(spec_methods.items()):
        low, high = details['speedup_range'].replace('x', '').split(' - ')
        low, high = float(low), float(high)
        mid = (low + high) / 2
        
        colors = []
        if details['vLLM'] == 'Yes':
            ax.barh(i - 0.15, mid, 0.3, xerr=[[mid-low], [high-mid]],
                   color='#2196F3', alpha=0.7, capsize=5, label='vLLM' if i == 0 else '')
        if details['SGLang'] in ('Yes', 'Limited'):
            alpha = 0.7 if details['SGLang'] == 'Yes' else 0.4
            ax.barh(i + 0.15, mid, 0.3, xerr=[[mid-low], [high-mid]],
                   color='#4CAF50', alpha=alpha, capsize=5, label='SGLang' if i == 0 else '')
    
    ax.set_yticks(range(len(methods)))
    ax.set_yticklabels(methods)
    ax.set_xlabel('Speedup (x)')
    ax.set_title('Speculative Decoding: Methods and Speedup Ranges', fontweight='bold')
    ax.axvline(x=1.0, color='black', linestyle='-', linewidth=1)
    ax.legend()
    
    plt.tight_layout()
    plt.show()

compare_speculative_decoding()

In [ ]:
# Demo 8: LoRA serving comparison

def compare_lora_serving():
    """Compare LoRA serving capabilities."""
    
    lora_features = [
        ('Max concurrent LoRAs', 'Configurable (default 8)', 'Configurable'),
        ('LoRA loading', 'Dynamic via API', 'Dynamic via API'),
        ('LoRA ranks supported', '8, 16, 32, 64, 128', '8, 16, 32, 64'),
        ('Multi-LoRA batching', 'Yes (SGMV/BGMV kernels)', 'Yes'),
        ('QLoRA support', 'Yes', 'Yes'),
        ('LoRA + quantization', 'Yes (limited combos)', 'Yes (limited combos)'),
        ('Per-request LoRA selection', 'Yes', 'Yes'),
    ]
    
    print("LoRA Serving Comparison")
    print(f"{'Feature':<35} {'vLLM':<30} {'SGLang':<30}")
    print('=' * 95)
    for feature, vllm, sglang in lora_features:
        print(f"{feature:<35} {vllm:<30} {sglang:<30}")
    
    # Simulated LoRA overhead comparison
    num_loras = [1, 4, 8, 16, 32]
    vllm_overhead = [2, 5, 8, 15, 25]  # % overhead vs no-LoRA
    sglang_overhead = [2, 5, 9, 16, 28]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(num_loras, vllm_overhead, 'o-', color='#2196F3', label='vLLM', linewidth=2)
    ax.plot(num_loras, sglang_overhead, 's-', color='#4CAF50', label='SGLang', linewidth=2)
    ax.set_xlabel('Number of Active LoRAs')
    ax.set_ylabel('Throughput Overhead (%)')
    ax.set_title('LoRA Serving Overhead vs Number of Active Adapters', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

compare_lora_serving()

In [ ]:
# Demo 9: Multi-modal support comparison

def compare_multimodal():
    """Compare multi-modal model support."""
    
    models = [
        ('LLaVA-1.5', 'Vision', True, True),
        ('LLaVA-NeXT', 'Vision', True, True),
        ('Qwen-VL', 'Vision', True, True),
        ('InternVL2', 'Vision', True, True),
        ('Phi-3-Vision', 'Vision', True, True),
        ('LLaVA-OneVision', 'Video', True, True),
        ('MiniCPM-V', 'Vision', True, True),
        ('CogVLM2-Video', 'Video', False, True),
        ('Pixtral', 'Vision', True, True),
    ]
    
    print("Multi-Modal Model Support")
    print(f"{'Model':<25} {'Type':<10} {'vLLM':<10} {'SGLang':<10}")
    print('=' * 55)
    
    vllm_count = 0
    sglang_count = 0
    for model, mtype, vllm, sglang in models:
        v_str = 'Yes' if vllm else 'No'
        s_str = 'Yes' if sglang else 'No'
        print(f"{model:<25} {mtype:<10} {v_str:<10} {s_str:<10}")
        if vllm: vllm_count += 1
        if sglang: sglang_count += 1
    
    print(f"\nTotal: vLLM={vllm_count}/{len(models)}, SGLang={sglang_count}/{len(models)}")
    
    # Visualization
    fig, ax = plt.subplots(figsize=(8, 5))
    categories = ['Vision Models', 'Video Models', 'Total']
    vision_v = sum(1 for m in models if m[1] == 'Vision' and m[2])
    vision_s = sum(1 for m in models if m[1] == 'Vision' and m[3])
    video_v = sum(1 for m in models if m[1] == 'Video' and m[2])
    video_s = sum(1 for m in models if m[1] == 'Video' and m[3])
    
    x = np.arange(len(categories))
    ax.bar(x - 0.2, [vision_v, video_v, vllm_count], 0.4, label='vLLM', color='#2196F3')
    ax.bar(x + 0.2, [vision_s, video_s, sglang_count], 0.4, label='SGLang', color='#4CAF50')
    ax.set_xticks(x)
    ax.set_xticklabels(categories)
    ax.set_ylabel('Models Supported')
    ax.set_title('Multi-Modal Model Support Count', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()

compare_multimodal()

---
## Hands-on Assignment 1: Build an Interactive Feature Comparison Tool

Create a tool that lets users filter and compare features based on their needs.

In [ ]:
# Assignment 1: Interactive feature comparison tool

class FeatureComparator:
    """Interactive tool for comparing vLLM and SGLang features."""
    
    def __init__(self, feature_matrix):
        self.features = feature_matrix
    
    def list_categories(self):
        """List all feature categories."""
        for i, cat in enumerate(self.features.keys()):
            print(f"  {i+1}. {cat}")
    
    def compare_category(self, category: str):
        """Compare all features in a category."""
        if category not in self.features:
            print(f"Category '{category}' not found.")
            return
        features = self.features[category]
        print(f"\n{category}:")
        print(f"{'Feature':<30} {'vLLM':<25} {'SGLang':<25}")
        print('-' * 80)
        for f, d in features.items():
            print(f"{f:<30} {d['vLLM']:<25} {d['SGLang']:<25}")
    
    def find_unique_features(self, system: str):
        """Find features unique to one system."""
        unique = []
        other = 'SGLang' if system == 'vLLM' else 'vLLM'
        for cat, features in self.features.items():
            for feat, details in features.items():
                if ('yes' in details[system].lower() and
                    ('no' in details[other].lower() or 'limited' in details[other].lower())):
                    unique.append((cat, feat, details[system]))
        return unique
    
    def score_for_use_case(self, priorities: Dict[str, float]) -> Dict[str, float]:
        """Score each system based on user priorities.
        
        priorities: dict of {category: weight} where weight is 0-10
        """
        # TODO: Implement scoring based on weighted feature coverage
        scores = {'vLLM': 0.0, 'SGLang': 0.0}
        # Hint: For each category, count supported features
        # and multiply by the priority weight
        return scores

# Usage
comparator = FeatureComparator(feature_matrix)

print("Feature Categories:")
comparator.list_categories()

print("\n\nvLLM Unique Features:")
for cat, feat, detail in comparator.find_unique_features('vLLM'):
    print(f"  [{cat}] {feat}: {detail}")

print("\nSGLang Unique Features:")
for cat, feat, detail in comparator.find_unique_features('SGLang'):
    print(f"  [{cat}] {feat}: {detail}")

# TODO: Implement score_for_use_case and test with:
# chatbot_priorities = {'Core Serving': 10, 'Frontend & Programming': 8, 'Quantization': 5}
# print(comparator.score_for_use_case(chatbot_priorities))

---
## Hands-on Assignment 2: Evaluate Both Systems for a Specific Use Case

Pick a use case and do a thorough evaluation.

In [ ]:
# Assignment 2: Use case evaluation

use_cases = {
    'Customer Support Chatbot': {
        'requirements': [
            'Low TTFT for responsiveness',
            'Multi-turn conversation support',
            'Structured output (JSON) for ticket creation',
            'Multiple LoRA adapters (per-department)',
            'High availability and scalability',
        ],
        'critical_features': ['Prefix Caching', 'Constrained Decoding',
                              'LoRA', 'Streaming Output'],
        'vllm_score': 0,  # TODO: Score 1-10
        'sglang_score': 0,  # TODO: Score 1-10
        'recommendation': 'TODO',
        'justification': 'TODO',
    },
    'Code Generation Service': {
        'requirements': [
            'Long context window (8k+ tokens)',
            'Fast prefill for large code files',
            'Speculative decoding for speed',
            'FIM (fill-in-the-middle) support',
            'Batch processing for CI/CD integration',
        ],
        'critical_features': ['Speculative Decoding', 'Prefix Caching',
                              'Batch Inference API', 'Chunked Prefill'],
        'vllm_score': 0,
        'sglang_score': 0,
        'recommendation': 'TODO',
        'justification': 'TODO',
    },
    'Data Extraction Pipeline': {
        'requirements': [
            'High throughput batch processing',
            'Structured JSON output (mandatory)',
            'Schema validation during generation',
            'Cost efficiency (quantization)',
            'Multiple schema support',
        ],
        'critical_features': ['Constrained Decoding', 'JSON Schema Decoding',
                              'Quantization', 'Batch Inference API'],
        'vllm_score': 0,
        'sglang_score': 0,
        'recommendation': 'TODO',
        'justification': 'TODO',
    },
}

for uc_name, uc in use_cases.items():
    print(f"\n{'='*60}")
    print(f"Use Case: {uc_name}")
    print(f"{'='*60}")
    print("\nRequirements:")
    for req in uc['requirements']:
        print(f"  - {req}")
    print(f"\nCritical Features: {', '.join(uc['critical_features'])}")
    print(f"\nScores: vLLM={uc['vllm_score']}/10, SGLang={uc['sglang_score']}/10")
    print(f"Recommendation: {uc['recommendation']}")
    print(f"Justification: {uc['justification']}")

print("\nTODO: Fill in scores, recommendations, and justifications for each use case.")

---
## Hands-on Assignment 3: Write a Recommendation Report

Create a structured recommendation report for a decision-maker.

In [ ]:
# Assignment 3: Recommendation report

def generate_recommendation_report():
    """Generate a structured recommendation report."""
    
    report = '''
    ============================================================
    RECOMMENDATION REPORT: vLLM vs SGLang
    ============================================================
    
    EXECUTIVE SUMMARY:
    TODO: Write a 2-3 sentence executive summary
    
    CHOOSE vLLM WHEN:
    - TODO: List 5 scenarios where vLLM is the better choice
    - 1. 
    - 2.
    - 3.
    - 4.
    - 5.
    
    CHOOSE SGLang WHEN:
    - TODO: List 5 scenarios where SGLang is the better choice
    - 1.
    - 2.
    - 3.
    - 4.
    - 5.
    
    FEATURE GAPS:
    - vLLM lacks: TODO
    - SGLang lacks: TODO
    
    RISK ASSESSMENT:
    - vLLM: TODO (community size, stability, roadmap)
    - SGLang: TODO (community size, stability, roadmap)
    '''
    print(report)
    
    # Generate a summary visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Strengths comparison
    strengths = {
        'vLLM Strengths': [
            ('Ecosystem breadth', 9),
            ('Quantization options', 9),
            ('Community size', 9),
            ('Production stability', 8),
            ('Documentation', 8),
        ],
        'SGLang Strengths': [
            ('Prefix caching', 10),
            ('Frontend DSL', 10),
            ('Constrained decoding', 9),
            ('Disaggregated serving', 8),
            ('Multi-turn efficiency', 9),
        ]
    }
    
    for ax, (title, items) in zip(axes, strengths.items()):
        names = [i[0] for i in items]
        scores = [i[1] for i in items]
        color = '#2196F3' if 'vLLM' in title else '#4CAF50'
        bars = ax.barh(names, scores, color=color, edgecolor='black', alpha=0.8)
        ax.set_xlim(0, 11)
        ax.set_title(title, fontweight='bold',
                    color='#1565C0' if 'vLLM' in title else '#2E7D32')
        for bar, score in zip(bars, scores):
            ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                   f'{score}/10', va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('recommendation_summary.png', dpi=150, bbox_inches='tight')
    plt.show()

generate_recommendation_report()

In [ ]:
print("\n--- End of Chapter 10.5 ---")
print("Next: Chapter 10.6 - When to Use What: Decision Framework")
print("\nKey takeaways:")
print("  1. vLLM has broader quantization and speculative decoding support")
print("  2. SGLang's frontend DSL is a unique differentiator")
print("  3. SGLang excels at constrained decoding with jump-forward")
print("  4. Both have strong core serving capabilities")
print("  5. Multi-modal support is converging between both systems")

## Summary

| Category | vLLM Advantage | SGLang Advantage |
|----------|---------------|------------------|
| **Quantization** | More formats (GGUF, SqueezeLLM) | Comparable core support |
| **Speculative Decoding** | More methods supported | Eagle support |
| **Constrained Decoding** | Via Outlines | Jump-forward (faster) |
| **Frontend** | Wider ecosystem integration | Native DSL with fork/join |
| **LoRA** | Comparable | Comparable |
| **Multi-modal** | Comparable | Slightly more video models |
| **Distributed** | Mature Ray integration | Built-in DP + disaggregation |